In [2]:
from langchain_core.documents import Document
with open("doc_sample.md", "r") as f:
    document = f.read()

lc_document_loader = [Document(page_content=document)]
lc_document_loader

[Document(metadata={}, page_content='%IMAGE_DESCRIPTION: The image displays a logo consisting of stylized text. The text appears to be "bud" with a small yellow circle above the \'u\'.%\n\n## Inference Acceleration for \n\n## Large Language Models on CPUs \n\n{jithinvg, dittops, adarsh.ms}@bud.studio \n\n## Abstract \n\nIn recent years, large language models have demonstrated remarkable performance across various natural language processing (NLP) tasks. However, deploying these models for realworld applications often requires efficient inference solutions to handle the computational demands. In this paper, we explore the utilization of CPUs for accelerating the inference of large language models. Specifically, we introduce a parallelized approach to enhance throughput by 1) Exploiting the parallel processing capabilities of modern CPU architectures, 2) Batching the inference request. Our evaluation shows the accelerated inference engine gives an **18-22x** improvement in the generated 

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

with open("doc_sample.md", "r") as f:
    document = f.read()

lc_document_loader = [Document(page_content=document)]

chunk_size = 1000
chunk_overlap = 0

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, 
    chunk_overlap=chunk_overlap,
    separators=[" "]
)
output = text_splitter.split_documents(lc_document_loader)

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader(
    "output.pdf",
    mode="single",
    extract_tables_settings={"return_as": "text"},
)

loaded_documents = loader.load()
loaded_documents

/tmp/ipykernel_9205/3763873875.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
/mnt/Linux/Projects/grag/grag_core/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
loaded_documents

[Document(metadata={'producer': 'cairo 1.18.0 (https://cairographics.org)', 'creator': '', 'creationdate': '2026-06-13T23:36:20+05:30', 'source': 'output.pdf', 'file_path': 'output.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20260613233620+05'30"}, page_content='170  PART TWO \nBiological Foundations of Development\n \n• \n• \n3. Half the nerve cells in the average baby’s brain die over the first few years of life.\n4. Most children walk when they are ready, and no amount of encouragement will \nenable a 6-month-old to walk alone.\n5. A person’s hormones have little effect on human growth and development until \npuberty.\n6. Emotional trauma can seriously impair the physical growth of young children, even \nthose who are adequately nourished, free from illness, and not physically abused.\nJot down your responses and we will see how you did on this “pretest” as we d

In [ ]:
from manu_index import ManuIndex
from inference import ONNXEmbedder
MODEL_DIR = 'onnx_models/embeddinggemma_300m/onnx/model_q4.onnx'
TOKENIZER_DIR = 'onnx_models/embeddinggemma_300m'
MAX_LENGTH = 768

def client():
    return None

embeddings = ONNXEmbedder(MODEL_DIR, TOKENIZER_DIR, MAX_LENGTH)
db = ManuIndex(embeddings=embeddings, client=client)
doc_list = db.search(query="What is the main topic of the document?", top_k=3)

In [13]:
import os
import json
import numpy as np
from inference import ONNXEmbedder
from sklearn.metrics.pairwise import cosine_similarity

MODEL_DIR = 'onnx_models/embeddinggemma_300m/onnx/model_q4.onnx'
TOKENIZER_DIR = 'onnx_models/embeddinggemma_300m'
MAX_LENGTH = 1024
META_FILENAME = "_meta.json"
persist_directory = "manu_index"
meta_path = os.path.join(persist_directory, META_FILENAME)

embedder = ONNXEmbedder(MODEL_DIR, TOKENIZER_DIR, MAX_LENGTH)
query = "What is the main topic of the document?"
query = embedder.embed_query(query)

with open(meta_path, "r") as f:
    try:
        data = json.load(f)
        # check the embedding shape of the first document to ensure it's correct
        print("Embedding size of the first document:", len(data[0]["values"]))
        print("Query embedding size:", len(query))
        print("Metadata embedding size:", len(data[0]["values"]))
        data.sort(key=lambda x: cosine_similarity([query], [x["values"]]), reverse=True)
        output = data[0]["doc_id"]  # Return the doc_id of the most relevant document
        print("Most relevant document ID:", output)
        print("Similarity score:", cosine_similarity([query], [data[0]["values"]])[0][0])
    except json.JSONDecodeError:
        raise ValueError("Metadata file is corrupted.")

Embedding size of the first document: 768
Query embedding size: 768
Metadata embedding size: 768
Most relevant document ID: e928ab
Similarity score: 0.6169974278434461


In [31]:
len(data[0]["values"][0])

768

In [ ]:
embedder = ONNXEmbedder(MODEL_DIR, TOKENIZER_DIR, MAX_LENGTH)
query = "What is the main topic of the document?"
query1 = embedder.encode([query])[0]


In [2]:
from langchain_community.vectorstores import FAISS
from inference import ONNXEmbedder

persist_directory = "manu_index"
MODEL_DIR = 'onnx_models/embeddinggemma_300m/onnx/model_q4.onnx'
TOKENIZER_DIR = 'onnx_models/embeddinggemma_300m'
MAX_LENGTH = 1024

embedder = ONNXEmbedder(MODEL_DIR, TOKENIZER_DIR, MAX_LENGTH)

vector_store = FAISS.load_local(
    folder_path=persist_directory,
    embeddings=embedder,
    index_name="9dce31",
    allow_dangerous_deserialization=True
)

In [3]:
# check the number of documents in the vector store
print("Number of documents in vector store:", len(vector_store.index_to_docstore_id))

Number of documents in vector store: 19


In [4]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "lambda_mult": 0.5,
    },
)

retriever_output = retriever.invoke("What is the main topic of the document?")
print(retriever_output)

[Document(id='db4e1a1b-fc49-4785-80ac-8cdb0db32986', metadata={}, page_content='Feb. 29, 2024. [Online]. Available: https://arxiv.org/abs/2401.02669v1 \n\n- [7] W. Kwon _et al._ , “Efficient Memory Management for Large Language Model Serving with PagedAttention,” _SOSP 2023 - Proceedings of the 29th ACM Symposium on Operating Systems Principles_ , vol. 1, pp. 611–626, Oct. 2023, doi: 10.1145/3600006.3613165. \n\n- [8] C. Lameter, “NUMA (Non-Uniform Memory Access): An Overview,” _Queue_ , vol. 11, no. 7, pp. 40–51, Jul. 2013, doi: 10.1145/2508834.2513149. \n\n- [9] R. Li _et al._ , “StarCoder: may the source be with you!,” May 2023, Accessed: Feb. 29, 2024. [Online]. Available: https://arxiv.org/abs/2305.06161v2 \n\n- [10] B. Rozière _et al._ , “Code Llama: Open Foundation Models for Code,” Aug. 2023, Accessed: Feb. 29, 2024. [Online]. Available: https://arxiv.org/abs/2308.12950v3'), Document(id='9b29e7b5-ec11-49f2-9eec-cdfa36b3744b', metadata={}, page_content='models on 4th Gen Intel® 